In [1]:
from Preprocess import *
from Train import *
import pandas as pd
import os
from pathlib import Path


Opening Annotations

In [26]:
annotations_path = Path('../../data/annotations')
csvs_path = annotations_path.glob('text*.csv')
annotations = []

for csv_path in csvs_path:
    csv = pd.read_csv(csv_path)
    annotations.append(csv)

print(len(annotations))
reader = CSVReader(annotations)
print(reader.entities.columns)
print(reader.entities_relations.columns)


12
Index(['data', 'label'], dtype='str')
Index(['e1_type', 'e2_type', 'intermediate_text', 'relation_label'], dtype='str')


Training Models

In [27]:
entity_classifier = EntityClassifier()
entity_classifier.fit(reader.entities['data'], reader.entities['label'])
relation_classifier = RelationClassifier()
relation_classifier.fit(reader.entities_relations['e1_type'], reader.entities_relations['e2_type'], reader.entities_relations['intermediate_text'], reader.entities_relations['relation_label'])

EntityClassifier trained successfully.
RelationClassifier trained successfully.


Testing

In [47]:
test = pd.Series(['the patient'])
ans = entity_classifier.predict(test)
print(ans)

0    FINDING
dtype: str


In [43]:
def parse_value_and_unit(text_str: str) -> tuple[str, str]:
    """
    Extrai o valor numérico e a unidade de medida para entidades do tipo VALUE.
    Exemplos:
      '6cm' -> ('6', 'cm')
      '12,476.5ng/ml' -> ('12476.5', 'ng/ml')
      '6cm x 9cm' -> ('6x9', 'cm')
      '9.5cm x 4.5cm x 2.0cm' -> ('9.5x4.5x2.0', 'cm')
    """
    # Identifica pares de (número, unidade) no texto
    matches = re.findall(r'([\d.,]+)\s*([a-zA-Z/%]+)?', text_str)
    if not matches:
        return "", ""
    
    nums = []
    unit = ""
    for val, u in matches:
        clean_num = val.replace(',', '')
        nums.append(clean_num)
        if u and not unit:
            unit = u
            
    val_str = "x".join(nums) if len(nums) > 1 else nums[0]
    return val_str, unit


def process_cases_to_graph_csvs(
    cases_df: pd.DataFrame,
    entity_classifier: EntityClassifier,
    relation_classifier: RelationClassifier,
    output_nodes_path: str = "nodes.csv",
    output_edges_path: str = "edges.csv",
    max_distance: int = 10,
    context_k: int = 3
):
    """
    Lê o DataFrame de casos clínicos (cases.csv) e gera os arquivos nodes.csv e edges.csv.
    """
    processor = MedicalTextProcessor()
    
    nodes_list = []
    edges_list = []
    
    global_node_counter = 1
    global_edge_counter = 1

    for _, case_row in cases_df.iterrows():
        case_id = case_row['case_id']
        raw_text = str(case_row['case_text'])

        # Divide o texto em períodos limpos
        sentences = processor.split_sentences(raw_text)

        global_token_offset = 0
        
        for sentence in sentences:
            # Localiza o offset do caractere do período no texto completo
            sent_char_start = raw_text.find(sentence)
            if sent_char_start == -1:
                sent_char_start = 0

            # Processa a frase extraindo o Doc, candidatos a entidades e pares
            doc, candidates, pairs = processor.process_sentence(
                sentence, 
                max_distance=max_distance, 
                context_k=context_k
            )

            if not candidates:
                global_token_offset += len(doc)
                continue

            # 1. Classificação de Entidades (Modelo 1)
            cand_series = pd.Series([c['text'] for c in candidates])
            predicted_types = entity_classifier.predict(cand_series)

            # Mapeia entidades válidas
            valid_entities_in_sent = {}

            for cand, p_type in zip(candidates, predicted_types):
                if p_type not in ['NOT_ENTITY', 'OUTSIDE']:
                    node_id = f"N_{global_node_counter:03d}"
                    global_node_counter += 1

                    # Calcula offsets absolutos no texto do caso
                    abs_span_start = sent_char_start + cand['start_char']
                    abs_span_end = sent_char_start + cand['end_char']
                    abs_token_start = global_token_offset + cand['start_token']
                    abs_token_end = global_token_offset + cand['end_token']

                    # Extrai value e unit se for tipo VALUE
                    val_str, unit_str = "", ""
                    if p_type == 'VALUE':
                        val_str, unit_str = parse_value_and_unit(cand['text'])

                    node_info = {
                        'node_id': node_id,
                        'case_id': case_id,
                        'node_type': p_type,
                        'label_original': cand['text'],
                        'label_normalizado': cand['text'].lower(),
                        'token_start': abs_token_start,
                        'token_end': abs_token_end,
                        'span_start': abs_span_start,
                        'span_end': abs_span_end,
                        'value': val_str,
                        'unit': unit_str
                    }

                    nodes_list.append(node_info)
                    
                    # Guarda objeto da entidade com o tipo para a fase de relações
                    cand['type'] = p_type
                    valid_entities_in_sent[cand['text']] = cand

            # 2. Classificação de Relações (Modelo 2)
            for pair in pairs:
                e1_text = pair['e1']['text']
                e2_text = pair['e2']['text']

                # Só avalia relação se ambas as entidades forem classificadas como válidas
                if e1_text in valid_entities_in_sent and e2_text in valid_entities_in_sent:
                    e1 = valid_entities_in_sent[e1_text]
                    e2 = valid_entities_in_sent[e2_text]

                    s_e1_type = pd.Series([e1['type']])
                    s_e2_type = pd.Series([e2['type']])
                    s_intermed = pd.Series([pair['intermediate_text']])

                    rel_label = relation_classifier.predict(s_e1_type, s_e2_type, s_intermed).iloc[0]

                    if rel_label != 'NO_RELATION':
                        edge_id = f"E_{global_edge_counter:03d}"
                        global_edge_counter += 1

                        edges_list.append({
                            'edge_id': edge_id,
                            'case_id': case_id,
                            'e1_normalizado': e1['text'].lower(),
                            'e1_text': e1['text'],
                            'e1_type': e1['type'],
                            'e2_normalizado': e2['text'].lower(),
                            'e2_text': e2['text'],
                            'e2_type': e2['type'],
                            'relation_label': rel_label,
                            'intermediate_text': pair['intermediate_text']
                        })

            # Atualiza o offset acumulado de tokens
            global_token_offset += len(doc)

    # Converte em DataFrames
    df_nodes = pd.DataFrame(nodes_list)
    df_edges = pd.DataFrame(edges_list)

    # Garante a ordem exata das colunas especificadas no projeto
    nodes_columns = [
        'node_id', 'case_id', 'node_type', 'label_original', 'label_normalizado',
        'token_start', 'token_end', 'span_start', 'span_end', 'value', 'unit'
    ]
    edges_columns = [
        'edge_id', 'case_id', 'e1_normalizado', 'e1_text', 'e1_type',
        'e2_normalizado', 'e2_text', 'e2_type', 'relation_label', 'intermediate_text'
    ]

    df_nodes = df_nodes.reindex(columns=nodes_columns)
    df_edges = df_edges.reindex(columns=edges_columns)

    # Salva os arquivos CSV
    df_nodes.to_csv(output_nodes_path, index=False)
    df_edges.to_csv(output_edges_path, index=False)

    print(f"Sucesso! Gerados:")
    print(f" - {output_nodes_path}: {len(df_nodes)} nós registrados.")
    print(f" - {output_edges_path}: {len(df_edges)} arestas registradas.")

    return df_nodes, df_edges



In [ ]:
cases_path = Path('../../data/raw/cases.csv')
cases = pd.read_csv(cases_path)
used_cases = cases.iloc[[0]]

process_cases_to_graph_csvs(cases_df= used_cases, entity_classifier= entity_classifier, relation_classifier= relation_classifier)

Sucesso! Gerados:
 - nodes.csv: 42 nós registrados.
 - edges.csv: 14 arestas registradas.
